In [5]:
code = '''"""
Lead-matched eval for the refit2023 + fx model.
fx features are recomputed AFTER the lead swap (no observed-future leak).
Compare against: 3.20% (tuned), 3.37% (floor), 3.96% (statewide).
"""
import os, json, gc, glob
import numpy as np, pandas as pd, torch
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

FOLDER    = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
FEATURES  = os.path.join(FOLDER, "zonal_features_fx.parquet")
LEADS     = os.path.join(FOLDER, "weather_leads_zonal.parquet")
CKPT      = os.path.join(FOLDER, "checkpoints_zonal", "zonal_tft_refit2023_fx_best.ckpt")
OUT_JSON  = os.path.join(FOLDER, "metrics_zonal_refit2023_fx.json")

TRAIN_START="2015-07-01 04:00"; VAL_START="2024-01-01 05:00"; TEST_START="2024-01-23 12:00"
ENCODER_LEN, DECODER_LEN = 168, 120
BATCH, SEED = 128, 42
HOT_DAY_C, CDH_BASE_C = 24.0, 22.0

ZONE_WEATHER = {"WEST":"A_WEST","GENESE":"B_GENESE","CENTRL":"C_CENTRL","NORTH":"D_NORTH",
    "MHK VL":"E_MHKVL","CAPITL":"F_CAPITL","HUD VL":"G_HUDVL","MILLWD":"G_HUDVL",
    "DUNWOD":"J_NYC","N.Y.C.":"J_NYC","LONGIL":"K_LONGIL"}
LEAN_VARS=["temperature_2m","apparent_temperature","relative_humidity_2m",
           "wind_speed_10m","shortwave_radiation","cloud_cover"]
FX=["fx_app_roll72","fx_cdh24","fx_hot_streak_day"]
UNKNOWN_REALS=["demand","demand_lag24","demand_lag168",
               "demand_roll24_mean","demand_roll168_mean","demand_roll24_std"]
KNOWN_REALS=LEAN_VARS+["temp_vshape"]+FX+["time_idx"]
KNOWN_CATS=["hour","day_of_week","month","is_weekend","is_holiday"]

def lead_file(d): return os.path.join(FOLDER, "fx_lead%d.npz" % d)

def add_fx(df):
    df = df.sort_values(["zone","utc"]).reset_index(drop=True)
    df["fx_app_roll72"] = (df.groupby("zone")["apparent_temperature"]
                             .transform(lambda s: s.rolling(72, min_periods=1).mean()))
    cdh = (df["apparent_temperature"] - CDH_BASE_C).clip(lower=0)
    df["fx_cdh24"] = cdh.groupby(df["zone"]).transform(lambda s: s.rolling(24, min_periods=1).sum())
    d = df["utc"].dt.date
    daily = df.assign(_d=d).groupby(["zone","_d"])["apparent_temperature"].max().rename("dmax").reset_index()
    daily["hot"] = daily["dmax"] >= HOT_DAY_C
    def streak(s):
        out, run = [], 0
        for h in s:
            run = run + 1 if h else 0
            out.append(run)
        return pd.Series(out, index=s.index)
    daily["fx_hot_streak_day"] = daily.groupby("zone")["hot"].transform(streak).astype(float)
    df = df.assign(_d=d).merge(daily[["zone","_d","fx_hot_streak_day"]],
                               on=["zone","_d"], how="left", suffixes=("_old",""))
    df = df.drop(columns=[c for c in df.columns if c.endswith("_old") or c == "_d"])
    return df

def load_base():
    df=pd.read_parquet(FEATURES); df["utc"]=pd.to_datetime(df["utc"])
    for c in KNOWN_CATS: df[c]=df[c].astype(str).astype("category")
    df["zone"]=df["zone"].astype(str)
    df=df[df["utc"]>=TRAIN_START].copy()
    df["time_idx"]=df["time_idx"]-df["time_idx"].min()
    return df.sort_values(["zone","time_idx"]).reset_index(drop=True)

def swap_lead(base, leads, d):
    out=base.copy(); boundary=pd.Timestamp(TEST_START); fb=0
    for zone,wkey in ZONE_WEATHER.items():
        mask=out["zone"]==zone; utc=out.loc[mask,"utc"]; post=utc>=boundary
        for v in LEAN_VARS:
            s=utc.map(leads["%s_prev_day%d__%s" % (v,d,wkey)]).interpolate(limit=6)
            s5=utc.map(leads["%s_prev_day5__%s" % (v,wkey)]).interpolate(limit=6)
            s=s.fillna(s5); fb+=int((s.isna()&post).sum())
            s=s.fillna(out.loc[mask,v]); out.loc[mask,v]=s.values
    out["temp_vshape"]=(out["temperature_2m"]-14.0).abs()
    out=add_fx(out)
    print("  fallback cells: %d%s" % (fb, "  <-- INVESTIGATE" if fb else ""), flush=True)
    return out

def build_training_ds(df):
    val_idx=int(df.loc[df["utc"]>=VAL_START,"time_idx"].min())
    return TimeSeriesDataSet(df[df["time_idx"]<val_idx],
        time_idx="time_idx",target="demand",group_ids=["zone"],
        max_encoder_length=ENCODER_LEN,max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,static_categoricals=["zone"],
        target_normalizer=GroupNormalizer(groups=["zone"]),
        add_relative_time_idx=True,add_target_scales=True,allow_missing_timesteps=False)

@torch.no_grad()
def predict_one_zone(model, training_ds, df_zone, tsi, device, debug=False):
    ds = TimeSeriesDataSet.from_dataset(training_ds, df_zone,
        min_prediction_idx=tsi, stop_randomization=True)
    dl = ds.to_dataloader(train=False, batch_size=BATCH, num_workers=0)
    yh, yt, tt = [], [], []
    for bi, (x, (y, _)) in enumerate(dl):
        xd = {k:(v.to(device) if torch.is_tensor(v) else v) for k,v in x.items()}
        raw = model(xd)
        p = raw["prediction"][..., raw["prediction"].shape[-1]//2].cpu().numpy()
        if debug and bi == 0:
            print("      [debug] pred_MW~%.0f true~%.0f" % (p[0,0], float(y[0,0])), flush=True)
        yh.append(p); yt.append(y.cpu().numpy())
        tt.append(x["decoder_time_idx"][:,0].cpu().numpy())
        del xd, raw, p
    del dl, ds; gc.collect()
    if device=="cuda": torch.cuda.empty_cache()
    return np.concatenate(yh), np.concatenate(yt), np.concatenate(tt)

def main():
    pl.seed_everything(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    df=load_base()
    missing=[c for c in KNOWN_REALS if c not in df.columns]
    if missing: raise SystemExit("missing columns: " + str(missing))

    leads=pd.read_parquet(LEADS); leads.index=pd.to_datetime(leads.index)
    if getattr(leads.index,"tz",None) is not None: leads.index=leads.index.tz_localize(None)

    tsi=int(df.loc[df["utc"]>=TEST_START,"time_idx"].min())
    training_ds=build_training_ds(df)
    model=TemporalFusionTransformer.load_from_checkpoint(CKPT).to(device).eval()

    one=df[df["zone"]==df["zone"].iloc[0]]
    month_arr=np.full(int(df["time_idx"].max())+DECODER_LEN+2,-1)
    month_arr[one["time_idx"].values]=one["utc"].dt.month.values

    df=df[df["time_idx"]>=tsi-ENCODER_LEN-1].copy()
    print("trimmed to %d rows, test_start_idx=%d, device=%s" % (len(df), tsi, device), flush=True)
    ZONES=sorted(df["zone"].unique())

    FLOOR=[2.48,2.89,3.14,3.52,3.96]
    day_mape, composite = {}, []
    for d in range(1,6):
        print("\\n=== Lead %d ===" % d, flush=True)
        if os.path.exists(lead_file(d)):
            z=np.load(lead_file(d)); ape,months=z["ape"],z["months"]; print("  cached",flush=True)
        else:
            df_d=swap_lead(df,leads,d)
            hat,true={},{}
            for zi,zone in enumerate(ZONES):
                yh,yt,tt=predict_one_zone(model,training_ds,df_d[df_d["zone"]==zone],tsi,device,
                                          debug=(d==1 and zi==0))
                for i,t in enumerate(tt):
                    hat.setdefault(int(t),[]).append(yh[i])
                    true.setdefault(int(t),[]).append(yt[i])
                print("    %-10s (%d/11)" % (zone, zi+1), flush=True)
            del df_d; gc.collect()
            good=sorted(t for t,v in hat.items() if len(v)==11)
            drop=len(hat)-len(good)
            if drop: print("  dropped %d incomplete windows" % drop, flush=True)
            sh=np.stack([np.sum(hat[t],axis=0) for t in good])
            st=np.stack([np.sum(true[t],axis=0) for t in good])
            ut=np.array(good); s,e=(d-1)*24,d*24
            ape=np.abs(st[:,s:e]-sh[:,s:e])/np.clip(st[:,s:e],1e-6,None)
            months=month_arr[ut[:,None]+np.arange(s,e)[None,:]]
            np.savez(lead_file(d),ape=ape,months=months)
            del hat,true; gc.collect()
        day_mape[d]=float(ape.mean()*100)
        print("Day %d: %.2f%%   (tuned %.2f%%)  [saved]" % (d, day_mape[d], FLOOR[d-1]), flush=True)
        composite.append((ape.ravel(),months.ravel()))

    all_ape=np.concatenate([a for a,_ in composite]); all_mon=np.concatenate([m for _,m in composite])
    overall=float(all_ape.mean()*100)
    smap={12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
          6:"Summer",7:"Summer",8:"Summer",9:"Fall",10:"Fall",11:"Fall"}
    seasons={}
    for sn in ["Winter","Spring","Summer","Fall"]:
        m=np.isin(all_mon,[k for k,v in smap.items() if v==sn])
        seasons[sn]=float(all_ape[m].mean()*100) if m.any() else None

    print("\\n===== REFIT2023 + FX RESULTS =====")
    for d in range(1,6):
        print("Day %d: %.2f%%   (tuned %.2f%%)" % (d, day_mape[d], FLOOR[d-1]))
    print("Overall: %.2f%%   (tuned 3.20%%, floor 3.37%%, statewide 3.96%%)" % overall)
    print("\\nSeasonal (tuned: Winter 3.35, Spring 3.37, Summer 3.60, Fall 2.35):")
    for sn,v in seasons.items():
        print("  %s: %.2f%%" % (sn, v) if v else "  %s: n/a" % sn)

    json.dump({"day_mape":{str(k):round(v,3) for k,v in day_mape.items()},
        "overall_mape":round(overall,3),
        "seasonal_mape":{k:(round(v,3) if v else None) for k,v in seasons.items()},
        "checkpoint":"zonal_tft_refit2023_fx_best.ckpt (2 epochs, lr=3e-4, +2023, +fx)"},
        open(OUT_JSON,"w"),indent=2)
    print("\\nSaved -> " + OUT_JSON)

if __name__ == "__main__":
    main()
'''

path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/06_eval_refit2023_fx.py"
with open(path, "w") as f:
    f.write(code)
print("written:", path, "|", len(code), "chars")

written: /opt/app-root/src/Forecasting-Energy-Demand/Sangar/06_eval_refit2023_fx.py | 9362 chars


In [3]:
import os
p = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/06_eval_refit2023_fx.py"
print(os.path.exists(p), os.path.getsize(p) if os.path.exists(p) else "")

True 9362


In [8]:
import subprocess, sys, os, time

ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"

print("GPU:")
print(subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_memory","--format=csv"],
                     capture_output=True, text=True).stdout)
print("already running:", subprocess.run(["pgrep","-af","06_eval_refit2023_fx"],
                                         capture_output=True, text=True).stdout.strip() or "none")

log = f"{ROOT}/eval_refit2023_fx.log"
with open(log, "w") as f:
    subprocess.Popen([sys.executable, "06_eval_refit2023_fx.py"],
                     stdout=f, stderr=subprocess.STDOUT, start_new_session=True, cwd=ROOT)
print("launched")
time.sleep(120)
print(subprocess.run(["tail","-15",log], capture_output=True, text=True).stdout)

GPU:
pid, used_gpu_memory [MiB]
5404, 250 MiB

already running: none
launched
===== REFIT2023 + FX RESULTS =====
Day 1: 2.21%   (tuned 2.48%)
Day 2: 2.54%   (tuned 2.89%)
Day 3: 2.86%   (tuned 3.14%)
Day 4: 3.32%   (tuned 3.52%)
Day 5: 3.81%   (tuned 3.96%)
Overall: 2.95%   (tuned 3.20%, floor 3.37%, statewide 3.96%)

Seasonal (tuned: Winter 3.35, Spring 3.37, Summer 3.60, Fall 2.35):
  Winter: 2.93%
  Spring: 3.11%
  Summer: 3.40%
  Fall: 2.27%

Saved -> /opt/app-root/src/Forecasting-Energy-Demand/Sangar/metrics_zonal_refit2023_fx.json



In [9]:
!grep -E "fallback|Lead|cached|dropped|debug" /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_refit2023_fx.log

=== Lead 1 ===
  cached
=== Lead 2 ===
  cached
=== Lead 3 ===
  cached
=== Lead 4 ===
  cached
=== Lead 5 ===
  cached


In [10]:
import subprocess, os
ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
print("--- fx_lead files ---")
print(subprocess.run(["ls","-l","--time-style=+%Y-%m-%d %H:%M:%S"] +
                     [f"{ROOT}/fx_lead{d}.npz" for d in range(1,6)],
                     capture_output=True, text=True).stdout)
print("--- eval script written at ---")
print(subprocess.run(["ls","-l","--time-style=+%Y-%m-%d %H:%M:%S",
                      f"{ROOT}/06_eval_refit2023_fx.py"], capture_output=True, text=True).stdout)
print("--- checkpoint written at ---")
print(subprocess.run(["ls","-l","--time-style=+%Y-%m-%d %H:%M:%S",
                      f"{ROOT}/checkpoints_zonal/zonal_tft_refit2023_fx_best.ckpt"],
                     capture_output=True, text=True).stdout)
print("--- full eval log ---")
print(subprocess.run(["cat", f"{ROOT}/eval_refit2023_fx.log"], capture_output=True, text=True).stdout[:2000])

--- fx_lead files ---
-rw-r--r--. 1 1000950000 1000950000 5907096 2026-07-23 15:23:22 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/fx_lead1.npz
-rw-r--r--. 1 1000950000 1000950000 5907096 2026-07-23 15:38:58 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/fx_lead2.npz
-rw-r--r--. 1 1000950000 1000950000 5907096 2026-07-23 16:08:38 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/fx_lead3.npz
-rw-r--r--. 1 1000950000 1000950000 5907096 2026-07-23 16:32:10 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/fx_lead4.npz
-rw-r--r--. 1 1000950000 1000950000 5907096 2026-07-23 16:45:50 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/fx_lead5.npz

--- eval script written at ---
-rw-r--r--. 1 1000950000 1000950000 9362 2026-07-23 15:24:42 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/06_eval_refit2023_fx.py

--- checkpoint written at ---
-rw-r--r--. 1 1000950000 1000950000 6173523 2026-07-23 13:36:17 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/checkpoints_zonal/zonal_

In [11]:
import os, subprocess, sys, time

ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
os.remove(f"{ROOT}/fx_lead1.npz")
print("removed fx_lead1.npz, leads 2-5 kept")

log = f"{ROOT}/eval_verify.log"
with open(log, "w") as f:
    subprocess.Popen([sys.executable, "06_eval_refit2023_fx.py"],
                     stdout=f, stderr=subprocess.STDOUT, start_new_session=True, cwd=ROOT)
print("launched, check back in ~15 min")

removed fx_lead1.npz, leads 2-5 kept
launched, check back in ~15 min


In [13]:
!grep -E "Lead 1|fallback|Day 1|debug" /opt/app-root/src/Forecasting-Energy-Demand/Sangar/eval_verify.log

=== Lead 1 ===
  fallback cells: 0
      [debug] pred_MW~1504 true~1518
Day 1: 2.21%   (tuned 2.48%)  [saved]
Day 1: 2.21%   (tuned 2.48%)


In [14]:
import subprocess, sys, os, getpass, importlib.util
if importlib.util.find_spec("mlflow") is None:
    subprocess.run([sys.executable,"-m","pip","install","-q","mlflow"])

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/Sangi2805/Forecasting-Energy-Demand.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "Sangi2805"
os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass.getpass("DagsHub token: ")

import json, mlflow
ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
OUT_JSON = f"{ROOT}/metrics_zonal_refit2023_fx.json"
m = json.load(open(OUT_JSON))

mlflow.set_experiment("Default")
with mlflow.start_run(run_name="tft_Sangar_zonal_refit2023_fx"):
    mlflow.log_params({
        "model": "TFT", "who": "Sangar", "data": "zonal_11zone_aggregated",
        "checkpoint": "zonal_tft_refit2023_fx_best.ckpt",
        "learning_rate": 3e-4, "reduce_on_plateau_patience": 2,
        "epochs_fixed": 2, "early_stopping": "none",
        "hidden_size": 64, "encoder_len": 168, "decoder_len": 120,
        "train_end": "2023-12-31", "val_window": "2024-01-01 to 2024-01-23",
        "test_start_utc": "2024-01-23 12:00", "protocol": "5pass_lead_matched",
        "features_added": "fx_app_roll72,fx_cdh24,fx_hot_streak_day",
        "n_known_reals": 11,
        "note": "TWO changes at once: +2023 in training AND fx heat features. Attribution not separable.",
    })
    mlflow.log_metric("overall_mape", m["overall_mape"])
    for d, v in m["day_mape"].items():
        mlflow.log_metric(f"day{d}_mape", v)
    for s, v in (m.get("seasonal_mape") or {}).items():
        if v is not None: mlflow.log_metric(f"{s.lower()}_mape", v)
    mlflow.log_metric("val_loss_best", 23.4116)
    mlflow.log_metric("prev_best_mape", 3.20)
    mlflow.log_metric("statewide_baseline_mape", 3.96)
    mlflow.log_artifact(OUT_JSON)
    print("logged run:", mlflow.active_run().info.run_id)

DagsHub token:  ········


logged run: 03280814f00945dc979d7fc29dfef944
🏃 View run tft_Sangar_zonal_refit2023_fx at: https://dagshub.com/Sangi2805/Forecasting-Energy-Demand.mlflow/#/experiments/0/runs/03280814f00945dc979d7fc29dfef944
🧪 View experiment at: https://dagshub.com/Sangi2805/Forecasting-Energy-Demand.mlflow/#/experiments/0


In [15]:
import subprocess, os
os.chdir("/opt/app-root/src/Forecasting-Energy-Demand")

files = ["Sangar/05_train_refit2023_fx.py",
         "Sangar/06_eval_refit2023_fx.py",
         "Sangar/metrics_zonal_refit2023_fx.json"]
subprocess.run(["git","add"]+files)
print(subprocess.run(["git","status","-s","--"]+files, capture_output=True, text=True).stdout)

r = subprocess.run(["git","commit","-m",
     "Zonal TFT refit through 2023 + heat-memory features: 2.95% lead-matched MAPE (was 3.20%)"],
     capture_output=True, text=True)
print(r.stdout or r.stderr)

r = subprocess.run(["git","push","sangar","main"], capture_output=True, text=True)
print(r.stdout, r.stderr)

A  Sangar/05_train_refit2023_fx.py
A  Sangar/06_eval_refit2023_fx.py
A  Sangar/metrics_zonal_refit2023_fx.json

[main e66d0bf] Zonal TFT refit through 2023 + heat-memory features: 2.95% lead-matched MAPE (was 3.20%)
 3 files changed, 313 insertions(+)
 create mode 100644 Sangar/05_train_refit2023_fx.py
 create mode 100644 Sangar/06_eval_refit2023_fx.py
 create mode 100644 Sangar/metrics_zonal_refit2023_fx.json

 To https://github.com/Sangi2805/Forecasting-Energy-Demand.git
   7b6ae28..e66d0bf  main -> main



In [18]:
import os, mlflow
out = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/runs.csv"
df = mlflow.search_runs(search_all_experiments=True)
df.to_csv(out, index=False)
print(len(df), "runs ->", out)

171 runs -> /opt/app-root/src/Forecasting-Energy-Demand/Sangar/runs.csv


In [19]:
# export_mlflow_runs.py
# Exports all MLflow runs from DagsHub to a single CSV backup.
# Run from /opt/app-root/src/Forecasting-Energy-Demand/Sangar/ on CAIR.

import os
import mlflow
import pandas as pd

TRACKING_URI = "https://dagshub.com/Sangi2805/Forecasting-Energy-Demand.mlflow"
OUT = "mlflow_runs_backup.csv"

# DagsHub auth. Set these two before running, or export them in your shell:
#   export MLFLOW_TRACKING_USERNAME=Sangi2805
#   export MLFLOW_TRACKING_PASSWORD=<your DagsHub token>
# Get the token from DagsHub, top right avatar, Settings, Tokens.
if not os.environ.get("MLFLOW_TRACKING_USERNAME"):
    os.environ["MLFLOW_TRACKING_USERNAME"] = "Sangi2805"
if not os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    raise SystemExit(
        "Set MLFLOW_TRACKING_PASSWORD to your DagsHub token first, then rerun."
    )

mlflow.set_tracking_uri(TRACKING_URI)
client = mlflow.tracking.MlflowClient()

experiments = client.search_experiments()
if not experiments:
    raise SystemExit("No experiments found. Check the tracking URI and token.")

frames = []
for exp in experiments:
    df = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        output_format="pandas",
        max_results=5000,
    )
    if len(df):
        df.insert(0, "experiment_name", exp.name)
        frames.append(df)
    print(f"{exp.name}: {len(df)} runs")

if not frames:
    raise SystemExit("Experiments exist but no runs were returned.")

runs = pd.concat(frames, ignore_index=True)

# Put the columns we care about first, keep everything else after.
lead = [c for c in [
    "experiment_name",
    "tags.mlflow.runName",
    "status",
    "start_time",
    "run_id",
] if c in runs.columns]
rest = [c for c in runs.columns if c not in lead]
runs = runs[lead + rest].sort_values("start_time")

runs.to_csv(OUT, index=False)
print(f"\nWrote {len(runs)} runs to {OUT}")

# Quick look at just the run names and any MAPE metric, for eyeballing.
mape_cols = [c for c in runs.columns if "mape" in c.lower()]
show = ["tags.mlflow.runName", "start_time"] + mape_cols
show = [c for c in show if c in runs.columns]
print(runs[show].to_string(index=False))

tft_leadmatched_full_hourly: 5 runs
tft_leadmatched: 15 runs
tft_masked: 1 runs
tft: 4 runs
sarimax: 1 runs
energy-demand-forecasting: 51 runs
Default: 94 runs

Wrote 171 runs to mlflow_runs_backup.csv
                                                                                                                 tags.mlflow.runName                       start_time  metrics.MAPE_pct  metrics.test_MAPE_pct_day1  metrics.train_MAPE_pct_day4  metrics.train_MAPE_pct_day5  metrics.val_MAPE_pct_day5  metrics.val_MAPE_pct_day4  metrics.val_MAPE_pct_day3  metrics.test_MAPE_pct_day4  metrics.train_MAPE_pct_day1  metrics.val_MAPE_pct_day2  metrics.train_MAPE_pct_day3  metrics.train_MAPE_pct_day2  metrics.val_MAPE_pct_day1  metrics.test_MAPE_pct_day5  metrics.test_MAPE_pct_day2  metrics.test_MAPE_pct_day3  metrics.MAPE_pct_day1  metrics.MAPE_pct_day3  metrics.MAPE_pct_day2  metrics.avg_mape  metrics.day2_mape  metrics.day3_mape  metrics.day1_mape  metrics.overall_mape  metrics.statewide_baseline_

In [ ]:
# step1_our_forecast.py — our Day-1 zonal forecast for one date, verified offline.
# Run on CAIR from the Sangar folder. No network needed for this step.
import importlib.util, os
import numpy as np, pandas as pd, torch

ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
EVAL = os.path.join(ROOT, "06_eval_refit2023_fx.py")

spec = importlib.util.spec_from_file_location("evalmod", EVAL)
ev = importlib.util.module_from_spec(spec); spec.loader.exec_module(ev)

def our_forecast(target_local_date):
    """Per-zone hourly Day-1 forecast (MW) for a local calendar date,
    plus the actual demand from the feature table on the same UTC hours."""
    df = ev.load_base()
    leads = pd.read_parquet(ev.LEADS); leads.index = pd.to_datetime(leads.index)
    if getattr(leads.index, "tz", None) is not None:
        leads.index = leads.index.tz_localize(None)

    one = df[df["zone"] == df["zone"].iloc[0]][["time_idx", "utc"]]
    idx2utc = dict(zip(one["time_idx"], one["utc"]))
    utc2idx = {v: k for k, v in idx2utc.items()}

    # local midnight of the target date -> UTC -> decoder start index
    start_utc = (pd.Timestamp(target_local_date, tz="America/New_York")
                   .tz_convert("UTC").tz_localize(None))
    if start_utc not in utc2idx:
        raise SystemExit(f"{start_utc} UTC not in table; pick a date in "
                         "2024-01-23 .. 2026-05-31.")
    dstart = utc2idx[start_utc]

    tsi = int(df.loc[df["utc"] >= ev.TEST_START, "time_idx"].min())
    training_ds = ev.build_training_ds(df)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = ev.TemporalFusionTransformer.load_from_checkpoint(ev.CKPT).to(device).eval()

    dfx = df[df["time_idx"] >= tsi - ev.ENCODER_LEN - 1].copy()
    dfx = ev.swap_lead(dfx, leads, 1)          # Day-1 forecast weather + fx after swap

    ZONES = sorted(dfx["zone"].unique())
    hours = pd.DatetimeIndex([idx2utc[dstart + h] for h in range(24)], name="utc")
    pred = pd.DataFrame(index=hours)
    for zone in ZONES:
        yh, yt, tt = ev.predict_one_zone(model, training_ds,
                                         dfx[dfx["zone"] == zone], tsi, device)
        w = np.where(tt == dstart)[0]
        if len(w) == 0:
            raise SystemExit(f"no window starts at {idx2utc[dstart]} for {zone}")
        pred[zone] = yh[w[0], :24]
    pred["STATEWIDE"] = pred[ZONES].sum(axis=1)

    act = (df[df["utc"].isin(hours)]
           .pivot_table(index="utc", columns="zone", values="demand", aggfunc="mean")
           .reindex(hours))
    act["STATEWIDE"] = act[ZONES].sum(axis=1)
    return pred, act

if __name__ == "__main__":
    D = "2025-07-15"     # a summer day inside the test window
    pred, act = our_forecast(D)
    mape = (np.abs(pred["STATEWIDE"] - act["STATEWIDE"]) / act["STATEWIDE"]).mean() * 100
    print(f"{D}  our statewide Day-1 MAPE vs actual: {mape:.2f}%")
    print(pred[["STATEWIDE"]].join(act["STATEWIDE"], rsuffix="_actual").round(0).to_string())

In [ ]:
for zi, zone in enumerate(ZONES):
        yh, yt, tt = ev.predict_one_zone(model, training_ds,
                                         dfx[dfx["zone"] == zone], tsi, device)
        w = np.where(tt == dstart)[0]
        if len(w) == 0:
            raise SystemExit(f"no window starts at {idx2utc[dstart]} for {zone}")
        pred[zone] = yh[w[0], :24]
        print(f"  {zone:10s} done ({zi+1}/11)", flush=True)